# 04 · Fontes extras

Grupos candidatos, um por célula (se uma célula der erro de conexão, rode só ela de novo). No fim, a correlação de todas as features com o alvo.

In [ ]:
import sys
from pathlib import Path

if Path("/kaggle/input").exists():
    RAIZ = next(Path("/kaggle/input").rglob("src/soja/__init__.py")).parents[2]
else:
    RAIZ = Path.cwd().parent
sys.path.insert(0, str(RAIZ / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from soja import config, dados, features, wasde

ETAPA = "04_extras"
PASTA = config.pasta_resultados(ETAPA)
plt.rcParams.update({"figure.figsize": (11, 3.5), "axes.grid": True, "grid.alpha": .3})

def ler(etapa, arquivo):
    df = pd.read_csv(config.ler_resultado(etapa, arquivo), index_col="mes")
    df.index = pd.PeriodIndex(df.index, freq="M")
    return df

preco = ler("01_preco", "preco.csv")

## 1. Mais revisões do WASDE

Produção da Argentina, exportações e esmagamento dos EUA, e estoque final do mundo: ln(este relatório ÷ anterior).

In [ ]:
cache = RAIZ / "data" / "wasde_tabelas.csv"
if not cache.exists():
    cache = config.DATA / "wasde_tabelas.csv"
tabelas = wasde.carregar_tabelas(cache, desde=config.INICIO)
g_wasde = features.features_wasde_extras(tabelas).loc[config.INICIO:]

g_wasde.plot(subplots=True, figsize=(11, 8), title="Revisões extras do WASDE (log)", legend=True, color=["tab:brown", "tab:red", "tab:orange", "tab:blue"])
plt.tight_layout(); plt.show()
g_wasde.tail()

## 2. Posição dos fundos (CFTC)

(compras − vendas dos fundos) ÷ contratos em aberto na soja de Chicago, no fim do mês.

In [ ]:
cot = dados.baixar_cot(ate=pd.Timestamp.today().year)
g_fundos = features.features_fundos(cot).loc[config.INICIO:]

fig, ax = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
g_fundos["fundos_liquido"].plot(ax=ax[0], color="tab:green", title="Posição líquida dos fundos (fração dos contratos)")
g_fundos["fundos_var"].plot(ax=ax[1], color="tab:gray", title="Variação no mês")
for a in ax: a.axhline(0, c="k", lw=.7)
plt.tight_layout(); plt.show()
g_fundos.tail()

## 3. Momentum longo e volatilidade

In [ ]:
g_momentum = features.features_momentum(preco["preco_usd_t"])

fig, ax = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
g_momentum[["tendencia_6m", "tendencia_12m"]].plot(ax=ax[0], title="Tendência de 6 e 12 meses (log)")
g_momentum["volatilidade_6m"].plot(ax=ax[1], color="tab:purple", title="Volatilidade de 6 meses")
ax[0].axhline(0, c="k", lw=.7); plt.tight_layout(); plt.show()
g_momentum.tail()

## 4. Exportações do Brasil (Comex Stat)

Soja em grão (NCM 12010090 até 2011 e 12019000 depois), comparada com o mesmo mês do ano anterior. Uso o mês anterior, porque o dado sai só no começo do mês seguinte.

In [ ]:
exportacao = dados.baixar_exportacoes_br(desde=1999, ate=pd.Timestamp.today().year)
g_export = features.features_exportacao(exportacao).loc[config.INICIO:]

fig, ax = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
(exportacao.loc[config.INICIO:] / 1e6).plot(ax=ax[0], color="tab:green", title="Exportação mensal (milhões de t)")
g_export["exportacao_br_12m"].plot(ax=ax[1], color="tab:olive", title="Contra o mesmo mês do ano anterior (log, mês t−1)")
ax[1].axhline(0, c="k", lw=.7); plt.tight_layout(); plt.show()
g_export.tail()

## 5. Fim do mês e fundos por tipo

- `fim_vs_media`: ln(último dia) − ln(média do mês), com o preço diário de Chicago (`ZS=F`)
- `ultima_semana`: variação nos últimos 5 pregões
- `gestores_*` e `produtores_var`: relatório desagregado da CFTC (só existe desde 2006)

In [ ]:
%pip install -q yfinance
diario = dados.baixar_soja_diaria()
g_fim = features.features_fim_do_mes(diario).loc[config.INICIO:]

cot_tipo = dados.baixar_cot_desagregado(ate=pd.Timestamp.today().year)
g_tipo = features.features_fundos_por_tipo(cot_tipo)

fig, ax = plt.subplots(3, 1, figsize=(11, 9), sharex=True)
g_fim["fim_vs_media"].plot(ax=ax[0], color="tab:green", title="Último dia contra a média do mês (log)")
g_fim["ultima_semana"].plot(ax=ax[1], color="tab:olive", title="Variação na última semana (log)")
g_tipo[["gestores_liquido"]].plot(ax=ax[2], color="tab:red", title="Posição líquida dos gestores de fundos")
for a in ax: a.axhline(0, c="k", lw=.7)
plt.tight_layout(); plt.show()
pd.concat([g_fim, g_tipo], axis=1).tail()

## 6. Correlação de tudo com o alvo (antes do teste)

Em cinza, as features das etapas 01 a 03; em cor, as novas.

In [ ]:
extras = pd.concat([g_wasde, g_fundos, g_momentum, g_export, g_fim, g_tipo], axis=1)
anteriores = pd.concat([
    preco.drop(columns=["preco_usd_t", "alvo"]),
    ler("02_enso", "oni.csv"),
    ler("03_wasde", "wasde.csv").drop(columns="sem_relatorio"),
], axis=1)

juntos = pd.concat([anteriores, extras], axis=1).join(preco["alvo"], how="inner")
juntos = juntos.loc[:pd.Period(config.TESTE_INICIO) - 1]
corr = juntos.drop(columns="alvo").corrwith(juntos["alvo"]).sort_values()

cores = ["tab:orange" if c in extras.columns else "lightgray" for c in corr.index]
corr.plot.barh(color=cores, figsize=(9, 8), title="Correlação com o alvo (2000 a 2014)")
plt.axvline(0, color="k", lw=.8); plt.tight_layout(); plt.savefig(PASTA / "correlacao.png", dpi=120); plt.show()
corr.round(3).to_frame("correlacao").T

## 7. Salvando

In [ ]:
saida = extras.loc[config.INICIO:].copy()
saida.index = saida.index.astype(str)
saida.index.name = "mes"
saida.to_csv(PASTA / "extras.csv")
corr.rename("correlacao").to_csv(PASTA / "correlacao.csv")
config.salvar_config(
    PASTA, fontes=["USDA WASDE via ESMIS", "CFTC COT legacy (005602)", "FMI (preço)", "Comex Stat NCM 12010090 e 12019000", "Yahoo Finance ZS=F", "CFTC desagregado (005602)"],
    colunas=list(extras.columns), meses=len(saida), ultimo_mes=saida.index[-1],
)
print(sorted(p.name for p in PASTA.iterdir()))